# Notebook 14 — Organization Size Effects and Model Robustness

**Purpose:** Investigate whether the supervised diversion-disclosure model disproportionately prioritizes large nonprofits, compare the model against peer-anomaly ranking within organization-size groups, and test transparent size-stratified alternatives. **This is a retrospective diagnostic, not a model retraining exercise or a validated fraud-detection system.**

**Inputs:** Corrected Notebook 12 organization-level queues. **Outputs:** `data/processed/notebook14/`. Run top to bottom in the `irs990` environment.

**Important:** The supervised model predicts historical self-reported significant diversion disclosures, **not independently verified criminal fraud**. Organization size can legitimately relate to disclosure opportunities and reporting; a size association alone does not prove unfairness or model error. Alternative rankings below are sensitivity analyses only and do not overwrite Notebook 12.

## 1. Configuration and input validation

Fail early on missing or inconsistent upstream data. Keep the original queue intact.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
INPUT_PRIMARY = DATA / 'notebook12' / '07_2024_organization_primary_queue.parquet'
INPUT_LIMITED = DATA / 'notebook12' / '07_2024_organization_limited_data_queue.parquet'
OUT = DATA / 'notebook14'
OUT.mkdir(parents=True, exist_ok=True)
for path in [INPUT_PRIMARY, INPUT_LIMITED]:
    assert path.is_file(), f'Missing Notebook 12 export: {path}'

p = pd.read_parquet(INPUT_PRIMARY)
l = pd.read_parquet(INPUT_LIMITED)
required = {'ein','tax_year','revenue','assets','supervised_diversion_score',
            'peer_adverse_6','consensus_score','organization_supervised_rank',
            'organization_peer_rank','organization_consensus_rank','peer_group',
            'six_indicator_count','integration_cohort'}
assert required.issubset(p.columns), f'Missing required fields: {sorted(required-set(p.columns))}'
assert p['ein'].notna().all() and p['ein'].is_unique
assert l['ein'].notna().all() and l['ein'].is_unique
assert set(p['ein']).isdisjoint(set(l['ein']))
assert p['tax_year'].eq(2024).all() and l['tax_year'].eq(2024).all()
assert p['integration_cohort'].eq('primary_review').all()
assert l['integration_cohort'].eq('limited_data').all()

def num(x):
    return pd.to_numeric(x, errors='coerce').astype('float64').replace([np.inf,-np.inf],np.nan)

def overlap(a,b,k):
    return len(set(a.head(k)['ein']) & set(b.head(k)['ein']))

for col in ['revenue','assets','supervised_diversion_score','peer_adverse_6','consensus_score',
            'organization_supervised_rank','organization_peer_rank','organization_consensus_rank']:
    p[col] = num(p[col])
assert p['organization_consensus_rank'].notna().all()
assert p['supervised_diversion_score'].notna().all()
assert p['peer_adverse_6'].notna().all()
assert p['revenue'].notna().all(), 'Revenue is required for size stratification; investigate missing values'
print('Primary:',f'{len(p):,}','Limited:',f'{len(l):,}')
print('Columns:',len(p.columns),'Output:',OUT)

## 2. Revenue distribution and size strata

Use **revenue deciles** to create similarly sized comparison groups. Nonpositive revenue receives a separate stratum; decile boundaries are learned only from the 2024 primary population. These are descriptive cohorts, not new fraud labels.

In [ ]:
p['revenue_nonpositive'] = p['revenue'].le(0)
positive = p['revenue'].gt(0)
assert positive.any(), 'No positive revenue records'
# Rank-based quantiles ensure approximately balanced groups even with tied revenue.
p['size_stratum'] = 'Nonpositive revenue'
p.loc[positive,'size_stratum'] = (
    'Revenue decile ' + (pd.qcut(p.loc[positive,'revenue'].rank(method='first'),10,labels=False)+1).astype(str)
)
order = ['Nonpositive revenue'] + [f'Revenue decile {i}' for i in range(1,11)]
p['size_stratum'] = pd.Categorical(p['size_stratum'], categories=order, ordered=True)
size_summary = p.groupby('size_stratum',observed=True).agg(
    organizations=('ein','size'),min_revenue=('revenue','min'),
    median_revenue=('revenue','median'),max_revenue=('revenue','max'),
    median_supervised_score=('supervised_diversion_score','median'),
    median_peer_score=('peer_adverse_6','median')
).reset_index()
size_summary.to_csv(OUT/'02_revenue_strata_summary.csv',index=False)
display(size_summary)
print('Nonpositive revenue organizations:',int(p['revenue_nonpositive'].sum()))

## 3. Association between organization size and risk signals

Use Spearman correlations (rank association) and median scores by size stratum. These diagnostics show association, not causation or predictive bias.

In [ ]:
p['log10_revenue_positive'] = np.where(p['revenue'].gt(0),np.log10(p['revenue'].clip(lower=1)),np.nan)
p['log10_assets_positive'] = np.where(p['assets'].gt(0),np.log10(p['assets'].clip(lower=1)),np.nan)
rows=[]
for size_col in ['revenue','assets','log10_revenue_positive','log10_assets_positive']:
    for score_col in ['supervised_diversion_score','peer_adverse_6','consensus_score']:
        x=p[[size_col,score_col]].dropna()
        rows.append({'size_variable':size_col,'signal':score_col,'n':len(x),
                     'spearman_rho':x[size_col].corr(x[score_col],method='spearman') if len(x)>2 else np.nan})
correlations=pd.DataFrame(rows)
display(correlations)
correlations.to_csv(OUT/'03_size_score_correlations.csv',index=False)
plot=size_summary.loc[size_summary['size_stratum'].astype(str).str.startswith('Revenue decile')]
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(plot['size_stratum'].astype(str),plot['median_supervised_score'],marker='o')
ax.set(title='Median supervised score by revenue decile',xlabel='Revenue decile',ylabel='Median supervised score')
ax.tick_params(axis='x',rotation=45)
fig.tight_layout(); fig.savefig(OUT/'03_median_supervised_score_by_decile.png',dpi=150); plt.show()

## 4. Who appears in the highest-ranked organizations?

Compare the composition of supervised, peer-anomaly, and combined top-K lists by revenue stratum, and calculate enrichment relative to the primary population. Large concentration differences warrant scrutiny but do not establish bias.

In [ ]:
TOP_K=[100,500,1000,5000]
methods={'supervised':'organization_supervised_rank',
         'peer':'organization_peer_rank','combined':'organization_consensus_rank'}
base=p.groupby('size_stratum',observed=True).size()
composition=[]
for method,col in methods.items():
    for k in TOP_K:
        top=p.nsmallest(k,col)
        counts=top.groupby('size_stratum',observed=True).size()
        for group in base.index:
            n=int(counts.get(group,0)); population=int(base[group]);
            composition.append({'method':method,'top_k':k,'size_stratum':str(group),
                'selected':n,'population':population,'share_of_top_k':n/k,
                'share_of_population':population/len(p),
                'enrichment_ratio':(n/k)/(population/len(p))})
composition=pd.DataFrame(composition)
composition.to_csv(OUT/'04_topk_size_composition.csv',index=False)
display(composition.query('top_k == 1000').pivot(index='size_stratum',columns='method',values='selected'))
fig,ax=plt.subplots(figsize=(10,4))
for method in methods:
    part=composition.query('top_k == 1000 and method == @method')
    ax.plot(part['size_stratum'],part['share_of_top_k'],marker='o',label=method)
ax.set(title='Revenue-stratum share of top 1,000',xlabel='Revenue stratum',ylabel='Share of selected organizations')
ax.tick_params(axis='x',rotation=45);ax.legend()
fig.tight_layout();fig.savefig(OUT/'04_top1000_size_composition.png',dpi=150);plt.show()

## 5. Within-size ranking: supervised and peer methods

Compare the two methods **within each revenue stratum**, rather than comparing organizations of radically different sizes. We compute within-stratum percentile ranks, then evaluate how frequently each method selects the same organizations within its own stratum. These percentiles are descriptive; they are not calibrated probabilities.

In [ ]:
# Percentile = 1 for highest signal within a revenue stratum.
for signal,stem in [('supervised_diversion_score','supervised'),('peer_adverse_6','peer')]:
    p[f'{stem}_within_size_pct']=p.groupby('size_stratum',observed=True)[signal].rank(pct=True,method='average')
    assert p[f'{stem}_within_size_pct'].between(0,1).all()
within=[]
for group,g in p.groupby('size_stratum',observed=True):
    for fraction in [0.01,0.05]:
        k=max(1,int(np.ceil(len(g)*fraction)))
        s=set(g.nlargest(k,'supervised_diversion_score')['ein'])
        a=set(g.nlargest(k,'peer_adverse_6')['ein'])
        within.append({'size_stratum':str(group),'organizations':len(g),
                       'top_fraction':fraction,'selected_each':k,'overlap':len(s&a),
                       'overlap_fraction':len(s&a)/k,
                       'score_spearman':g['supervised_diversion_score'].corr(g['peer_adverse_6'],method='spearman')})
within=pd.DataFrame(within)
display(within)
within.to_csv(OUT/'05_within_size_method_agreement.csv',index=False)

## 6. Size-stratified ranking alternatives (sensitivity only)

Create two alternatives without retraining: (A) rank supervised scores **within** revenue strata, then rank the within-stratum percentiles globally; (B) equally combine within-stratum supervised and peer percentiles. Both can increase representation across size strata, but neither is validated as more effective for criminal investigations. The original Notebook 12 ranking is preserved.

In [ ]:
# Larger percentile means stronger relative signal.
p['alternative_supervised_size_pct']=p['supervised_within_size_pct']
p['alternative_combined_size_pct']=0.5*p['supervised_within_size_pct']+0.5*p['peer_within_size_pct']
for col,rank in [('alternative_supervised_size_pct','alternative_supervised_size_rank'),
                 ('alternative_combined_size_pct','alternative_combined_size_rank')]:
    order_idx=p.sort_values([col,'ein'],ascending=[False,True],kind='mergesort').index
    p[rank]=pd.Series(np.arange(1,len(p)+1),index=order_idx).reindex(p.index).astype('int64')
    assert p[rank].is_unique and p[rank].between(1,len(p)).all()
original=p.sort_values('organization_consensus_rank')
original_supervised=p.sort_values('organization_supervised_rank')
alt_supervised=p.sort_values('alternative_supervised_size_rank')
alt_combined=p.sort_values('alternative_combined_size_rank')
sensitivity=[]
for k in TOP_K:
    for baseline_name,baseline,alternative_name,alternative in [
        ('original supervised',original_supervised,'size-stratified supervised',alt_supervised),
        ('original combined',original,'size-stratified combined',alt_combined)]:
        common=overlap(baseline,alternative,k)
        sensitivity.append({'top_k':k,'baseline':baseline_name,'alternative':alternative_name,
                            'overlap':common,'replaced':k-common,'overlap_pct':common/k*100})
sensitivity=pd.DataFrame(sensitivity)
display(sensitivity)
sensitivity.to_csv(OUT/'06_size_adjustment_topk_sensitivity.csv',index=False)

## 7. Alternative ranking composition and stability

Compare representation across revenue strata and inspect how size-adjusted rankings change selected organizations. These changes are not evidence of improved fraud detection.

In [ ]:
alternatives={'original_combined':'organization_consensus_rank',
              'size_stratified_supervised':'alternative_supervised_size_rank',
              'size_stratified_combined':'alternative_combined_size_rank'}
alt_comp=[]
for method,col in alternatives.items():
    for k in [100,500,1000]:
        top=p.nsmallest(k,col)
        counts=top.groupby('size_stratum',observed=True).size()
        for group in base.index:
            alt_comp.append({'method':method,'top_k':k,'size_stratum':str(group),
                             'selected':int(counts.get(group,0))})
alt_comp=pd.DataFrame(alt_comp)
alt_comp.to_csv(OUT/'07_alternative_ranking_composition.csv',index=False)
display(alt_comp.query('top_k == 1000').pivot(index='size_stratum',columns='method',values='selected'))

## 8. Signal agreement, outliers, and data quality

Check whether the highest-ranked organizations are affected by missing financial values, nonpositive revenue, extreme assets or liabilities, or low indicator coverage. Flag potential issues without treating outliers as errors or misconduct.

In [ ]:
s=set(p.nsmallest(1000,'organization_supervised_rank')['ein'])
a=set(p.nsmallest(1000,'organization_peer_rank')['ein'])
p['signal_group']=np.select([p['ein'].isin(s&a),p['ein'].isin(s),p['ein'].isin(a)],
                           ['Both top 1000','Supervised only','Peer only'],default='Neither')
for c in ['assets','revenue','liabilities','expenses','net_income']:
    if c in p.columns: p[c]=num(p[c])
flags=[]
for c in ['revenue','assets','liabilities','expenses']:
    if c in p.columns:
        threshold=p[c].quantile(.999)
        p[f'{c}_extreme_high']=p[c].gt(threshold)
        flags.append(f'{c}_extreme_high')
p['missing_assets']=p['assets'].isna()
p['low_indicator_coverage']=num(p['six_indicator_count']).lt(6)
quality=[]
for method,col in alternatives.items():
    for k in [100,500,1000]:
        top=p.nsmallest(k,col)
        item={'method':method,'top_k':k,'median_revenue':top['revenue'].median(),
              'median_assets':top['assets'].median(),
              'nonpositive_revenue':int(top['revenue_nonpositive'].sum()),
              'missing_assets':int(top['missing_assets'].sum()),
              'fewer_than_six_indicators':int(top['low_indicator_coverage'].sum())}
        item.update({f'{flag}_count':int(top[flag].sum()) for flag in flags})
        quality.append(item)
quality=pd.DataFrame(quality)
display(quality)
quality.to_csv(OUT/'08_topk_data_quality.csv',index=False)
group_summary=p.groupby('signal_group').agg(organizations=('ein','size'),
    median_revenue=('revenue','median'),median_assets=('assets','median'),
    median_supervised_score=('supervised_diversion_score','median'),
    median_peer_score=('peer_adverse_6','median')).reset_index()
display(group_summary)
group_summary.to_csv(OUT/'08_signal_group_financial_summary.csv',index=False)

## 9. Export comparative research queue and reproducibility manifest

Retain original Notebook 12 scores/ranks alongside the alternatives. These outputs are restricted research artifacts, **not recommended operational replacements**. Do not publish identified organizations or describe them as fraudulent.

In [ ]:
export_cols=['ein','OBJECTID','tax_year','size_stratum','revenue','assets','peer_group',
 'six_indicator_count','supervised_diversion_score','peer_adverse_6','consensus_score',
 'organization_supervised_rank','organization_peer_rank','organization_consensus_rank',
 'supervised_within_size_pct','peer_within_size_pct',
 'alternative_supervised_size_rank','alternative_combined_size_rank','signal_group',
 'revenue_nonpositive','missing_assets','low_indicator_coverage']+flags
export_cols=[c for c in export_cols if c in p.columns]
research=p[export_cols].copy()
research['size_stratum']=research['size_stratum'].astype('string')
research['interpretation_notice']='Exploratory size sensitivity; no confirmed fraud labels or operational validation.'
research.to_parquet(OUT/'09_2024_size_robustness_research_queue.parquet',index=False)
research.nsmallest(1000,'alternative_combined_size_rank').to_csv(
    OUT/'09_size_stratified_top1000_research_preview.csv',index=False)
manifest={'notebook':'14 — Organization Size Effects and Model Robustness',
 'source_primary':str(INPUT_PRIMARY),'source_limited':str(INPUT_LIMITED),
 'primary_organizations':int(len(p)),'limited_organizations':int(len(l)),
 'stratification':'2024 primary positive-revenue rank deciles; nonpositive revenue separate',
 'ranking_alternatives':['within-size supervised percentile','50/50 within-size supervised and peer percentiles'],
 'cautions':['No model retraining','No confirmed fraud outcome','No external test-set validation',
             'Original Notebook 12 rankings unchanged','Size effects are associations, not proof of bias']}
(OUT/'09_manifest.json').write_text(json.dumps(manifest,indent=2))
assert research['ein'].is_unique
assert len(research)==len(p)
print('Wrote Notebook 14 artifacts to',OUT)
print('Research queue rows:',len(research))

## 10. Final Results, Interpretation, and Next Steps

### 10.1 Overview

Notebook 14 investigated whether organization size influences the supervised machine-learning model and the resulting investigation rankings.

Previous analyses identified substantial differences in organization size between nonprofits prioritized by the supervised model and those prioritized by peer-based anomaly detection.

The primary objectives were to:

1. Measure relationships between organization size and model scores.
2. Determine whether the highest-ranked organizations are disproportionately concentrated in particular revenue groups.
3. Compare supervised and peer-anomaly rankings within similarly sized organizations.
4. Evaluate size-stratified ranking alternatives without retraining the models.
5. Assess whether size adjustment improves ranking balance or introduces additional data-quality concerns.

The analysis used the **329,440 organizations in the primary investigation population** from Notebook 12.

All alternative rankings were evaluated as sensitivity analyses. The original rankings were preserved.

### 10.2 Revenue Distribution and Organization Size

Organizations were divided into ten revenue deciles based on positive annual revenue.

An additional group was created for organizations reporting zero or negative revenue.

| Population | Organizations |
|---|---:|
| Primary investigation population | 329,440 |
| Positive-revenue organizations | 326,308 |
| Nonpositive-revenue organizations | 3,132 |

Each positive-revenue decile contained approximately 32,630 organizations.

This approach allowed comparisons among nonprofits with broadly similar revenue levels while retaining organizations with nonpositive revenue as a separate group.

### 10.3 Relationship Between Organization Size and Model Scores

Spearman correlations were calculated between financial size measures and the supervised, peer-anomaly, and combined scores.

| Financial measure | Supervised score | Peer-anomaly score | Combined score |
|---|---:|---:|---:|
| Revenue | 0.099 | 0.061 | 0.100 |
| Assets | -0.304 | -0.000 | -0.180 |

Revenue exhibited only a weak positive relationship with the supervised model score.

Assets exhibited a moderate negative relationship with the supervised score, while the peer-anomaly score showed essentially no overall association with assets.

**Interpretation:** Organization size influences the model rankings, but the relationship is not a simple increase in risk score as revenue or assets increase.

These correlations are descriptive and do not establish causation or determine which features drive the model's predictions.

### 10.4 Concentration of Large Organizations in the Supervised Ranking

Although the overall revenue correlation was weak, the highest-ranked supervised organizations were heavily concentrated in the largest revenue decile.

| Ranking method | Top 1,000 organizations in largest revenue decile | Percentage |
|---|---:|---:|
| Supervised model | 654 | 65.4% |
| Peer anomalies | 96 | 9.6% |
| Original combined ranking | 50 | 5.0% |

The largest revenue decile represents approximately 10% of the primary population.

Its representation among the supervised top 1,000 is therefore substantially higher than its population share.

This indicates that organization size is an important characteristic of the supervised model's highest-ranked organizations, even though the overall relationship between revenue and model scores is weak.

**Key finding:** Population-wide correlations alone are insufficient to evaluate whether a ranking disproportionately prioritizes particular organization sizes.

### 10.5 Differences Between Supervised and Peer-Anomaly Rankings

The supervised and peer-anomaly approaches continued to identify substantially different organizations.

Only **32 organizations** appeared in the top 1,000 of both approaches.

The financial characteristics of the selected organizations differed considerably.

| Signal classification | Organizations | Median annual revenue |
|---|---:|---:|
| Supervised only | 968 | $407,274,665 |
| Peer anomalies only | 968 | $649,912 |
| Both approaches | 32 | $474,459 |
| Neither approach | 327,472 | $541,854 |

The supervised-only group had substantially higher median revenue than the other groups.

However, differences between the methods persisted even when organizations were compared within the same revenue decile.

For example, in the largest revenue decile, each approach selected its top 327 organizations, but only two organizations appeared in both selections.

The within-decile score correlation was approximately 0.168.

**Interpretation:** Organization size does not fully explain the disagreement between supervised and peer-anomaly rankings.

The methods appear to capture different financial and governance characteristics, although their investigative usefulness has not been established.

### 10.6 Size-Stratified Ranking Alternatives

Two alternative rankings were developed:

**Size-stratified supervised ranking**

Organizations were ranked according to their supervised model scores relative to other nonprofits within the same revenue stratum.

**Size-stratified combined ranking**

The supervised and peer-anomaly signals were evaluated using within-stratum comparisons to create an alternative combined ranking.

Neither alternative required retraining the underlying supervised model.

The alternatives were compared against the original rankings.

| Ranking comparison | Top 100 overlap | Top 500 overlap | Top 1,000 overlap |
|---|---:|---:|---:|
| Original vs. size-stratified supervised | 11 | 113 | 410 |
| Original vs. size-stratified combined | 81 | 429 | 873 |

The size-stratified supervised ranking replaced:

- 89 of the original top 100 organizations.
- 387 of the original top 500 organizations.
- 590 of the original top 1,000 organizations.

The size-stratified combined ranking was more stable but still replaced 127 organizations in the original top 1,000.

**Conclusion:** Adjusting rankings for organization size can materially change investigative priorities.

However, these changes do not demonstrate improved fraud detection or investigative usefulness.

### 10.7 Effects of Size Adjustment on Data Quality

The alternative rankings were also evaluated for indicator completeness and extreme financial values.

| Top 1,000 diagnostic | Original combined | Size-stratified combined | Size-stratified supervised |
|---|---:|---:|---:|
| Fewer than six available indicators | 156 | 127 | 352 |
| Extremely high revenue | 7 | 1 | 43 |
| Extremely high assets | 3 | 2 | 50 |
| Nonpositive revenue | 17 | 5 | 10 |

The size-stratified supervised ranking selected substantially more organizations with incomplete indicator coverage and extreme financial values.

This creates a tradeoff.

Size adjustment may reduce the concentration of large organizations, but it may also prioritize organizations with unusual financial structures, incomplete information, or potentially unreliable comparisons.

The size-stratified combined ranking showed fewer of these concerns, but its investigative effectiveness remains untested.

**Conclusion:** A more balanced distribution across organization sizes is not sufficient evidence that a ranking is better.

Alternative rankings must also be evaluated for data quality, interpretability, and investigative usefulness.

### 10.8 Key Findings

**Finding 1: Organization size affects the composition of supervised priorities.**

The supervised top 1,000 is heavily concentrated in the largest revenue decile, despite a weak overall correlation between revenue and model scores.

**Finding 2: Organization size does not fully explain model disagreement.**

Supervised and peer-anomaly methods continue to identify substantially different organizations even within similar revenue groups.

**Finding 3: Size adjustment materially changes investigative priorities.**

The size-stratified supervised ranking replaced 590 organizations in the original top 1,000.

**Finding 4: Size-adjusted rankings are not automatically better.**

The alternative supervised ranking selected more organizations with incomplete indicator coverage and extreme financial values.

**Finding 5: Model development has reached a practical evaluation stage.**

Further adjustments to rankings cannot establish which organizations are meaningful investigative leads without stronger outcome evidence or structured investigative review.

### 10.9 Remaining Limitations

Several limitations remain:

1. **No confirmed fraud outcomes:** The project cannot currently measure precision against independently established criminal fraud cases.

2. **Imperfect supervised target:** The supervised model predicts patterns associated with self-reported significant diversions of assets, not confirmed fraud.

3. **Descriptive size analysis:** Correlations and ranking concentrations do not establish that organization size causally drives model predictions.

4. **Unvalidated size adjustment:** The size-stratified rankings were not evaluated against independently verified investigative outcomes.

5. **Revenue-based grouping:** Organizations with similar revenue may differ substantially in mission, operating structure, funding sources, and financial practices.

6. **Extreme-value sensitivity:** Unusual financial values may reflect legitimate organizational circumstances or reporting errors.

7. **Limited governance information:** Form 990 data cannot capture every characteristic relevant to potential financial misconduct.

8. **No completed investigator assessments:** The project has not yet demonstrated that the highest-ranked organizations represent actionable investigative leads.

The rankings should continue to be interpreted as exploratory analytical priorities rather than probabilities of fraud or evidence of misconduct.

### 10.10 Decision: Retain Existing Rankings

Based on the Notebook 14 results, the original supervised, peer-anomaly, and combined rankings should remain unchanged.

The size-stratified alternatives should be retained as comparative research outputs.

There is insufficient evidence to conclude that either alternative provides better investigative prioritization.

Further model adjustments should be driven by demonstrated improvements in investigative usefulness rather than by ranking balance alone.

### 10.11 Next Steps: Notebook 15 — Investigative Case Studies and Lead Quality

The next phase will shift from model development toward evaluating whether the system identifies meaningful investigative leads.

Notebook 15 should:

1. Select representative organizations from the existing investigation queues, including organizations prioritized by both approaches and those prioritized by only one.
2. Generate detailed organization-level case profiles using financial histories, governance indicators, peer comparisons, and model explanations.
3. Identify the specific patterns that caused each organization to be flagged.
4. Highlight potential legitimate explanations, missing information, and data-quality concerns.
5. Develop structured review questions to help investigators assess whether additional examination is warranted.
6. Incorporate the blinded investigator review sample created in Notebook 13.
7. Prepare review-ready outputs that can eventually be assessed by qualified investigators.

The purpose is not to automatically classify organizations as fraudulent.

Instead, the goal is to determine whether the analytical system produces understandable, potentially useful leads that justify further investigation.

### 10.12 Final Conclusion

Notebook 14 demonstrated that organization size materially affects the composition of the supervised model's highest-ranked organizations.

However, organization size does not fully explain the differences between supervised and peer-anomaly methods.

Size-stratified rankings substantially change investigative priorities, but they also introduce potential tradeoffs involving incomplete indicators and extreme financial values.

The analysis does not establish that size-adjusted rankings improve investigative usefulness.

**Decision:** Retain the original Notebook 12 rankings, preserve size-stratified alternatives for research, and conclude the current model-robustness phase.

Proceed to Notebook 15 to evaluate investigative case quality, explainability, and practical usefulness.

The project's overarching objective remains developing a defensible analytical system that helps criminal investigators identify and prioritize nonprofit organizations exhibiting potentially suspicious financial or governance patterns.

The next critical question is no longer simply which organizations receive the highest model scores, but whether those organizations represent meaningful investigative leads.